# VTT Meeting Transcript → Embedding-Ready Chunks Pipeline

**Objective:** Convert raw WebVTT meeting transcript files stored in a UC volume into structured, deterministic chunks suitable for embedding and RAG retrieval.

## Architecture (Medallion)

| Layer | Table | Purpose |
|-------|-------|--------|
| **Bronze** | `dev.callagent.bronze.vtt_raw` | Raw VTT file content, one row per file |
| **Silver** | `dev.callagent.silver.utterances` | Parsed utterances: speaker, timestamps, text, PII-redacted |
| **Gold** | `dev.callagent.gold.transcript_chunks` | Deterministic chunks ready for embedding |

## Deterministic Chunking Strategy

1. Parse VTT into ordered utterances (speaker, start/end time, text)
2. Accumulate consecutive utterances into a chunk until `target_chars` is reached
3. Start a new chunk, carrying the **last utterance** of the previous chunk as overlap context
4. Each chunk gets a deterministic `chunk_id` = `SHA256(call_id || chunk_index)`
5. Chunk text is formatted with speaker labels

In [0]:
import re
import hashlib
import json
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# ── Catalog & Schema ──────────────────────────────────────────────
CATALOG = "dev"
BRONZE_SCHEMA = "callagent_bronze"
SILVER_SCHEMA = "callagent_silver"
GOLD_SCHEMA   = "callagent_gold"

# ── Source Volume ─────────────────────────────────────────────────
VOLUME_ROOT = f"/Volumes/{CATALOG}/callagent/raw/teams_transcripts"

# ── Table Names ──────────────────────────────────────────────────
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.vtt_raw"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.utterances"
GOLD_TABLE   = f"{CATALOG}.{GOLD_SCHEMA}.transcript_chunks"

# ── Chunking Parameters ──────────────────────────────────────────
TARGET_CHARS   = 2000   # target chunk size in characters (~500 tokens)
MIN_CHUNK_CHARS = 200   # don't emit a chunk smaller than this
OVERLAP_UTTS    = 1     # number of utterances to carry as overlap

# ── Create schemas if needed ─────────────────────────────────────
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

print("✅ Configuration loaded")
print(f"   Source volume : {VOLUME_ROOT}")
print(f"   Bronze table  : {BRONZE_TABLE}")
print(f"   Silver table  : {SILVER_TABLE}")
print(f"   Gold table    : {GOLD_TABLE}")
print(f"   Chunk target  : {TARGET_CHARS} chars (~{TARGET_CHARS // 4} tokens)")


## Bronze Layer: Raw VTT Ingestion

Reads `.vtt` files from the volume directory tree (partitioned by `call_date=YYYY-MM-DD`).
Each file becomes one row with the full file content, file path, and metadata.

> **Production note:** Replace the batch read with Auto Loader (`cloudFiles` format) for incremental/streaming ingestion.
> The batch version below is used for the initial draft and validation.

In [0]:
# ── Batch read all VTT files from the volume ─────────────────────
# Each file is read as a single row with `value` = full file text
raw_files = (
    spark.read.format("text")
    .option("wholetext", "true")
    .load(f"{VOLUME_ROOT}/**/*.vtt")
    .withColumnRenamed("value", "file_content")
    .withColumn("file_path", F.col("_metadata.file_path"))
)

# Extract call_date from partition path: .../call_date=2026-08-04/...
# Extract call_id from filename (deterministic hash of file path)
bronze_df = (
    raw_files
    .withColumn("call_date", F.regexp_extract(F.col("file_path"), r"call_date=(\d{4}-\d{2}-\d{2})", 1).cast("date"))
    .withColumn("file_name", F.regexp_extract(F.col("file_path"), r"([^/]+)\.vtt$", 1))
    #.withColumn("file_name", F.regexp_replace(F.regexp_replace(F.regexp_replace(F.regexp_extract(F.col("file_path"), r"([^/]+)\.vtt$", 1), "%20", " "), "%5B", "["), "%5D", "]"))
    .withColumn("call_id", F.sha2(F.col("file_path"), 256))
    .withColumn("ingestion_ts", F.current_timestamp())
    .select(
        "call_id",
        "call_date",
        "file_name",
        "file_path",
        "file_content",
        "ingestion_ts",
    )
)

print(f"Raw VTT files found: {bronze_df.count()}")
bronze_df.display()

In [0]:
# ── Write bronze table (full refresh for draft; use MERGE in production) ──
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)
print(f"✅ Bronze table written: {BRONZE_TABLE}")

## Silver Layer: Structured Utterances

Parses the WebVTT format into one row per utterance with:
- `call_id`, `call_date` (from bronze)
- `utterance_idx` — sequential index within the call (0-based)
- `speaker` — extracted from `<v Speaker Name>` tags
- `start_time` / `end_time` — original VTT timestamp strings
- `start_seconds` / `end_seconds` — numeric seconds for ordering/filtering
- `utterance_text` — clean text without VTT markup

In [0]:
# ── VTT Parsing Logic ────────────────────────────────────────────
# WebVTT format:
#   WEBVTT\n\n
#   00:00:00.000 --> 00:00:14.143\n
#   <v Speaker Name>Utterance text</v>\n\n
#   00:00:14.843 --> 00:00:33.700\n
#   <v Speaker Name>More text</v>\n

TS_PATTERN = re.compile(
    r"(\d{2}):(\d{2}):(\d{2})\.(\d{3})\s*-->\s*(\d{2}):(\d{2}):(\d{2})\.(\d{3})"
)
SPEAKER_PATTERN = re.compile(r"<v\s+([^>]+)>(.*?)</v>", re.DOTALL)

# ── PII Redaction Patterns ───────────────────────────────────────
PII_PATTERNS = [
    # Email addresses
    (re.compile(r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b"), "[EMAIL]"),
    # US phone numbers ((XXX) XXX-XXXX or XXX-XXX-XXXX)
    (re.compile(r"\(?\b\d{3}\)?[\s.\-]?\d{3}[\s.\-]?\d{4}\b"), "[PHONE]"),
    # Sensitive numbers (groups of 4 digits like the end of payment cards or account ids)
    (re.compile(r"\b(?:\d{4}[\s\-]?){3}\d{4}\b"), "[SENSITIVE_NUMBERS]"),
    # IPv4 addresses
    (re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"), "[IP_ADDR]"),
    # ZIP codes (5 digits, but avoid matching within longer numbers)
    (re.compile(r"\b\d{5}(?:\-\d{4})?\b"), "[ZIP]"),
]

def _redact_pii(text: str) -> str:
    """Mask common PII patterns in text with placeholders."""
    if not text:
        return text
    redacted = text
    for pattern, replacement in PII_PATTERNS:
        redacted = pattern.sub(replacement, redacted)
    return redacted


def _ts_to_seconds(h: str, m: str, s: str, ms: str) -> float:
    """Convert timestamp components to total seconds."""
    return int(h) * 3600 + int(m) * 60 + int(s) + int(ms) / 1000.0


def parse_vtt(file_content: str) -> list:
    """
    Parse a WebVTT transcript string into a list of utterance dicts.
    Returns: [{"utterance_idx", "speaker", "start_time", "end_time",
               "start_seconds", "end_seconds", "utterance_text"}, ...]
    """
    if not file_content or not file_content.strip():
        return []

    # Split into blocks separated by blank lines
    blocks = re.split(r"\n\s*\n", file_content.strip())

    utterances = []
    idx = 0

    for block in blocks:
        block = block.strip()
        if not block or block.startswith("WEBVTT"):
            continue

        # Find the timestamp line
        ts_match = TS_PATTERN.search(block)
        if not ts_match:
            continue

        start_time = f"{ts_match.group(1)}:{ts_match.group(2)}:{ts_match.group(3)}.{ts_match.group(4)}"
        end_time   = f"{ts_match.group(5)}:{ts_match.group(6)}:{ts_match.group(7)}.{ts_match.group(8)}"
        start_sec  = _ts_to_seconds(*ts_match.group(1, 2, 3, 4))
        end_sec    = _ts_to_seconds(*ts_match.group(5, 6, 7, 8))

        # Extract the text portion (everything after the timestamp line)
        text_part = block[ts_match.end():].strip()

        # Try to extract speaker from <v Speaker> tags
        speaker_match = SPEAKER_PATTERN.search(text_part)
        if speaker_match:
            speaker = speaker_match.group(1).strip()
            utterance_text = speaker_match.group(2).strip()
        else:
            # No speaker tag — clean any remaining VTT markup
            speaker = "Unknown"
            utterance_text = re.sub(r"<[^>]+>", "", text_part).strip()

        if utterance_text:
            redacted_text = _redact_pii(utterance_text)
            has_pii = redacted_text != utterance_text
            utterances.append({
                "utterance_idx":  idx,
                "speaker":        speaker,
                "start_time":     start_time,
                "end_time":       end_time,
                "start_seconds":  start_sec,
                "end_seconds":    end_sec,
                "utterance_text":  redacted_text,
                "pii_flagged":     has_pii,
            })
            idx += 1

    return utterances


# Register as a Spark UDF returning an array of structs
UTTERANCE_SCHEMA = T.ArrayType(T.StructType([
    T.StructField("utterance_idx",  T.IntegerType(),  False),
    T.StructField("speaker",        T.StringType(),   False),
    T.StructField("start_time",     T.StringType(),   False),
    T.StructField("end_time",       T.StringType(),   False),
    T.StructField("start_seconds",  T.DoubleType(),  False),
    T.StructField("end_seconds",    T.DoubleType(),  False),
    T.StructField("utterance_text",  T.StringType(),   False),
    T.StructField("pii_flagged",    T.BooleanType(),  False),
]))

parse_vtt_udf = F.udf(parse_vtt, UTTERANCE_SCHEMA)

print("✅ VTT parser registered as UDF")

# Quick sanity test on the sample file
sample_content = spark.table(BRONZE_TABLE).limit(1).collect()[0]["file_content"]
sample_parsed = parse_vtt(sample_content)
print(f"\nSample: parsed {len(sample_parsed)} utterances")
for u in sample_parsed[:3]:
    print(f"  [{u['start_time']} → {u['end_time']}] {u['speaker']}: {u['utterance_text'][:80]}...")

In [0]:
# ── ai_mask demo: LLM-based PII masking ──────────────────────────
# ai_mask(content, array('label1','label2',...)) uses a foundation model
# to detect and mask named entities. It replaces matched text with [MASKED].
#
# Supported labels include: person, email, phone, address, ssn, credit_card, etc.
# Full docs: https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-ai-functions/ai_mask

# 1) Quick standalone test
spark.sql("""
    SELECT ai_mask(
      'Hi, this is John Doe. My email is john.doe@example.com, call me at 555-123-4567.',
      array('person', 'email', 'phone')
    ) AS masked_text
""").display()

# 2) Apply to a sample of silver utterances (if the table exists)
try:
    spark.sql(f"""
        SELECT
            call_id,
            utterance_idx,
            speaker,
            utterance_text                                         AS original_text,
            ai_mask(utterance_text, array('person','email','phone','credit_card','ssn','address')) AS masked_text
        FROM {SILVER_TABLE}
        LIMIT 5
    """).display()
except Exception as e:
    print(f"Silver table not available yet: {e}")

In [0]:
# ── Silver: parse VTT and explode into one row per utterance with PII masking ───
silver_df = (
    spark.table(BRONZE_TABLE)
    .withColumn("parsed_utterances", parse_vtt_udf(F.col("file_content")))
    .withColumn("utterance", F.explode(F.col("parsed_utterances")))
    .select(
        F.col("call_id"),
        F.col("call_date"),
        F.col("file_name"),
        F.col("utterance.utterance_idx").alias("utterance_idx"),
        F.col("utterance.speaker").alias("speaker"),
        F.col("utterance.start_time").alias("start_time"),
        F.col("utterance.end_time").alias("end_time"),
        F.col("utterance.start_seconds").alias("start_seconds"),
        F.col("utterance.end_seconds").alias("end_seconds"),
        F.col("utterance.utterance_text").alias("utterance_text"),
        F.col("utterance.pii_flagged").alias("pii_flagged"),
    )
    # Mask PII using LLM-based semantic reasoning
    # .withColumn("masked_text", F.expr(f"ai_mask(utterance_text, array('email','phone','address','payment_card','account_id'))"))
    .orderBy("call_id", "utterance_idx")
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)
print(f"✅ Silver table written: {SILVER_TABLE}")
print(f"   Total utterances: {spark.table(SILVER_TABLE).count()}")
spark.table(SILVER_TABLE).display()

## Gold Layer: Deterministic Embedding Chunks

Groups ordered utterances into chunks using a **deterministic sliding-window** strategy:

1. Utterances are ordered by `utterance_idx` within each call
2. Consecutive utterances are accumulated until `TARGET_CHARS` is reached
3. A new chunk begins, carrying the last `OVERLAP_UTTS` utterance(s) from the previous chunk as overlap context
4. Each chunk is formatted as rich text with speaker labels
5. `chunk_id` = `SHA256(call_id || ':' || chunk_index)`

**Deterministic Chunking:**
- No ML-based or semantic splitting — purely character-count and utterance-boundary driven
- Same input always produces identical chunks, chunk indices, and chunk IDs
- Chunk text includes speaker metadata for richer embeddings

In [0]:
# ── Deterministic Chunking Logic ─────────────────────────────────
# Operates on a sorted list of utterances for a single call.
# Returns a list of chunk dicts.
def format_chunk_text(utterances: list) -> str:
    """
    Format a list of utterance dicts into plain text for embedding.
    Example output:
      Jordan Ruiz: Thanks for hopping on, Priya...
      Priya Nair: Sure. Right now it's a set of cron jobs...
    """
    lines = []
    for u in utterances:
        lines.append(f"{u['speaker']}: {u['utterance_text']}")
    return "\n".join(lines)
    
CHUNK_SCHEMA = T.ArrayType(T.StructType([
    T.StructField("chunk_id",         T.StringType(),  False),
    T.StructField("chunk_index",       T.IntegerType(), False),
    T.StructField("chunk_text",        T.StringType(),  False),
    T.StructField("speakers",           T.StringType(),  False),
    T.StructField("time_start",        T.StringType(),  False),
    T.StructField("time_end",          T.StringType(),  False),
    T.StructField("utterance_start_idx", T.IntegerType(), False),
    T.StructField("utterance_end_idx",   T.IntegerType(), False),
    T.StructField("utterance_count",   T.IntegerType(), False),
    T.StructField("char_count",        T.IntegerType(), False),
    T.StructField("token_estimate",    T.IntegerType(), False),
    T.StructField("metadata_json",     T.StringType(),  False),
]))


def chunk_call(utterances: list, call_id: str,
               target_chars: int = TARGET_CHARS,
               overlap_utts: int = OVERLAP_UTTS,
               min_chars: int = MIN_CHUNK_CHARS) -> list:
    """
    Deterministic chunking of a call's utterances.
    
    Args:
        utterances: list of dicts sorted by utterance_idx
        call_id:    unique call identifier
        target_chars: target chunk size in characters
        overlap_utts:  number of trailing utterances to carry into next chunk
        min_chars:  minimum chunk size to emit
    
    Returns: list of chunk dicts matching CHUNK_SCHEMA
    """
    if not utterances:
        return []

    # Sort by utterance_idx (defensive — caller should already sort)
    utterances = sorted(utterances, key=lambda u: u["utterance_idx"])

    chunks = []
    current = []
    current_chars = 0
    chunk_index = 0

    for utt in utterances:
        utt_chars = len(utt["utterance_text"])

        # If adding this utterance exceeds target and current chunk is non-trivial,
        # emit the current chunk and start a new one with overlap
        if (current_chars + utt_chars > target_chars
                and current_chars >= min_chars
                and current):
            # Emit current chunk
            chunk_text = format_chunk_text(current)
            speakers = ", ".join(sorted(set(u["speaker"] for u in current)))
            time_start = current[0]["start_time"]
            time_end = current[-1]["end_time"]
            utt_start = current[0]["utterance_idx"]
            utt_end = current[-1]["utterance_idx"]

            chunk_id = hashlib.sha256(
                f"{call_id}:{chunk_index}".encode("utf-8")
            ).hexdigest()

            metadata = json.dumps({
                "call_id": call_id,
                "chunk_index": chunk_index,
                "speakers": speakers,
                "time_start": time_start,
                "time_end": time_end,
                "utterance_range": [utt_start, utt_end],
                "utterance_count": len(current),
                "char_count": current_chars,
                "token_estimate": current_chars // 4,
            }, sort_keys=True)

            chunks.append({
                "chunk_id":            chunk_id,
                "chunk_index":         chunk_index,
                "chunk_text":          chunk_text,
                "speakers":            speakers,
                "time_start":          time_start,
                "time_end":            time_end,
                "utterance_start_idx": utt_start,
                "utterance_end_idx":   utt_end,
                "utterance_count":     len(current),
                "char_count":          current_chars,
                "token_estimate":      current_chars // 4,
                "metadata_json":       metadata,
            })

            # Start new chunk with overlap
            overlap = current[-overlap_utts:] if overlap_utts > 0 else []
            current = list(overlap)  # copy
            current_chars = sum(len(u["utterance_text"]) for u in current)
            chunk_index += 1

        # Add current utterance
        current.append(utt)
        current_chars += utt_chars

    # Emit final chunk
    if current and current_chars >= min_chars:
        chunk_text = format_chunk_text(current)
        speakers = ", ".join(sorted(set(u["speaker"] for u in current)))
        time_start = current[0]["start_time"]
        time_end = current[-1]["end_time"]
        utt_start = current[0]["utterance_idx"]
        utt_end = current[-1]["utterance_idx"]

        chunk_id = hashlib.sha256(
            f"{call_id}:{chunk_index}".encode("utf-8")
        ).hexdigest()

        metadata = json.dumps({
            "call_id": call_id,
            "chunk_index": chunk_index,
            "speakers": speakers,
            "time_start": time_start,
            "time_end": time_end,
            "utterance_range": [utt_start, utt_end],
            "utterance_count": len(current),
            "char_count": current_chars,
            "token_estimate": current_chars // 4,
        }, sort_keys=True)

        chunks.append({
                "chunk_id":            chunk_id,
                "chunk_index":         chunk_index,
                "chunk_text":          chunk_text,
                "speakers":            speakers,
                "time_start":          time_start,
                "time_end":            time_end,
                "utterance_start_idx": utt_start,
                "utterance_end_idx":   utt_end,
                "utterance_count":     len(current),
                "char_count":          current_chars,
                "token_estimate":      current_chars // 4,
                "metadata_json":       metadata,
            })

    return chunks


# Test on sample data
sample_call_id = spark.table(BRONZE_TABLE).limit(1).collect()[0]["call_id"]
sample_utterances = [
    row.asDict() for row in
    spark.table(SILVER_TABLE)
    .filter(F.col("call_id") == sample_call_id)
    .orderBy("utterance_idx")
    .collect()
]
sample_chunks = chunk_call(sample_utterances, sample_call_id)
print(f"Sample: {len(sample_utterances)} utterances → {len(sample_chunks)} chunks")
for c in sample_chunks:
    print(f"  Chunk {c['chunk_index']}: {c['char_count']} chars, {c['utterance_count']} utts, speakers=[{c['speakers']}]")
    print(f"    ID: {c['chunk_id'][:16]}...")
    print(f"    Text preview: {c['chunk_text'][:120]}...")
    print()